# speech playground

record your voice -> transcribe (stt), design a voice -> generate speech (tts).
reads `RUNPOD_API_KEY`, `STT_ENDPOINT_ID`, `TTS_ENDPOINT_ID` from `.env`.

In [ ]:
import base64, os, io
from pathlib import Path

import requests
from IPython.display import Audio, display

for line in Path(".env").read_text().splitlines():
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())

API = "https://api.runpod.ai/v2"
HEADERS = {"Authorization": f"Bearer {os.environ['RUNPOD_API_KEY']}"}


def call(endpoint_id, input_payload):
    resp = requests.post(
        f"{API}/{endpoint_id}/runsync", headers=HEADERS,
        json={"input": input_payload}, timeout=600,
    )
    resp.raise_for_status()
    output = resp.json().get("output") or {}
    if output.get("error"):
        raise RuntimeError(output["error"])
    return output


def transcribe(audio_bytes, language=None):
    payload = {"audio": base64.b64encode(audio_bytes).decode()}
    if language:
        payload["language"] = language
    return call(os.environ["STT_ENDPOINT_ID"], payload)


def speak(text, instruct="", language="Auto", **sampling):
    output = call(os.environ["TTS_ENDPOINT_ID"],
                  {"text": text, "instruct": instruct, "language": language, **sampling})
    audio = base64.b64decode(output["audio"])
    display(Audio(audio, rate=output["sample_rate"]))
    return audio


def record(seconds=5, rate=16000):
    import sounddevice as sd
    import soundfile as sf
    print(f"recording {seconds}s... speak now")
    audio = sd.rec(int(seconds * rate), samplerate=rate, channels=1, dtype="int16")
    sd.wait()
    buffer = io.BytesIO()
    sf.write(buffer, audio, rate, format="WAV")
    return buffer.getvalue()


print("ready")

## 1. record your voice and transcribe it

In [ ]:
my_voice = record(seconds=5)
display(Audio(my_voice, rate=16000))
transcribe(my_voice)

## 2. design a voice and listen

In [ ]:
speak(
    "Hi! This voice was designed on demand, just for you.",
    instruct="friendly young woman, upbeat and warm, medium pace",
    language="English",
)

In [ ]:
# another voice - same text, different character
speak(
    "Hi! This voice was designed on demand, just for you.",
    instruct="elderly man, gravelly voice, slow and thoughtful",
    language="English",
    temperature=0.8,
)

## 3. full loop: design a voice for your own transcript

In [ ]:
text = transcribe(my_voice)["text"]
print(f"you said: {text}")
speak(text, instruct="deep movie-trailer narrator, dramatic, slow", language="English")